In [64]:
# ============================================================
# INPUT SYMBOLS AS A LIST WITH ANCHOR SYMBOL LAST
# ============================================================

sorted_symbols_list = ['SCHH', 'USRT']

In [ ]:
# --- system setup ---
import sys
import os
from pathlib import Path 
sys.path.append(os.path.abspath(".."))

# from datetime import date
import asyncio
import numpy as np
import pandas as pd

from ib_insync import *
from ibkr.Class_IBKR_IB import IBKR_IB
ibkr = IBKR_IB(port=7496)

async def start_ibkr():
    await ibkr.connect()
    print("IBKR connected:", ibkr.ib.isConnected())



In [ ]:
lookback_period = "5 Y"
length_of_each_period = "1 day"
use_regular_trading_hours = True
prices_to_use = "TRADES"

In [70]:
async def get_historical_closes_df(contract_list, 
                                   lookback_period, 
                                   length_of_each_period='1 day',
                                   prices_to_use='TRADES',
                                   use_regular_trading_hours=True):

    df_list = []

    for contract in contract_list:

        sym = contract.symbol

        bars = await ibkr.ib.reqHistoricalDataAsync(
            contract=contract,
            endDateTime="",          # "" means now
            durationStr=lookback_period,
            barSizeSetting=length_of_each_period,
            whatToShow=prices_to_use,
            useRTH=use_regular_trading_hours,
            formatDate=1
        )

        df = pd.DataFrame([(bar.date, bar.close) for bar in bars], columns=["date", "close"])
        df['date'] = pd.to_datetime(df['date']).dt.date
        df[sym] = df['close']
        df = df.set_index("date")
        
        df_list.append(df[sym])
        
    big_df = pd.concat(df_list, axis=1)
    big_df = big_df.iloc[:-1]

    return big_df


In [71]:

async def main():

    await start_ibkr()

    contract_list = []
    
    for sym in sorted_symbols_list:
        contract = Stock(sym, 'SMART', 'USD')
        await ibkr.ib.qualifyContractsAsync(contract)
        contract_list.append(contract)

    df = await get_historical_closes_df(contract_list, lookback_period)

    df.reset_index(names="to date", inplace=True)
    df.insert(0, "from date", df['to date'].shift(1))

    for sym in sorted_symbols_list:
        from_col_name = f"from {sym} price"
        to_col_name = f"to {sym} price"     

        df[from_col_name] = df[sym].shift(1)
        df[to_col_name] = df[sym]

        df[f'{sym} cod'] = df[to_col_name] - df[from_col_name]
        df[f'{sym} pct cod'] = np.log(df[to_col_name] / df[from_col_name])

        df.drop(sym, axis =1, inplace=True)

    anchor = sorted_symbols_list[-1]
    for sym in sorted_symbols_list:
        df[f"to {anchor} / to {sym}"] = df[f"to {anchor} price"] / df[f"to {sym} price"]

    filename = "_".join(sorted_symbols_list)
    filename = filename + ".csv"
    file_path = Path("enhanced_prices/") / filename
    
    df.to_csv(file_path)

    print()
    print('finished')
    print()


In [72]:
# ============================================================
# MAIN
# ============================================================ 

await main()


IBKR connected: True

finished

